# Phân Tích Thời Gian Chờ (Bank Wait Time vs Customer Wait Time)

Trong file này, chúng ta sẽ bóc tách:
- **Customer Wait Time (Khách hàng ngâm):** Thời gian từ lúc ngân hàng gửi Offer (`O_Sent (mail and online)`) đầu tiên đến lúc nhận được phản hồi từ khách (`O_Returned`) đầu tiên.
- **Bank Wait Time (Ngân hàng duyệt):** Thời gian từ lúc nhận phản hồi (`O_Returned`) đến lúc ngân hàng đưa ra phán quyết sau khi thẩm định (Có thể là `A_Incomplete` - thiếu hồ sơ, `A_Pending` - chờ giải ngân, `A_Accepted`, hoặc `A_Denied`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình biểu đồ
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# 1. Đọc dữ liệu
data_path = 'data/bpi-challenge-2017/bpi_2017_cleaned.csv'
df = pd.read_csv(data_path)

# Chuyển đổi timestamp sang dạng datetime (UTC), xử lý format mixed
df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], utc=True, format='mixed')

# Sắp xếp dữ liệu theo thứ tự thời gian của từng case
df = df.sort_values(by=['case:concept:name', 'time:timestamp'])

print(f"Tổng số events: {len(df)}")
print(f"Tổng số cases: {df['case:concept:name'].nunique()}")

In [ ]:
# 2. Trích xuất thời điểm quan trọng cho từng Case

# 2.1 Customer Wait: Lấy O_Sent và O_Returned đầu tiên
o_sent = df[df['concept:name'] == 'O_Sent (mail and online)'].groupby('case:concept:name')['time:timestamp'].min().reset_index(name='time_sent')
o_returned = df[df['concept:name'] == 'O_Returned'].groupby('case:concept:name')['time:timestamp'].min().reset_index(name='time_returned')

# 2.2 Bank Wait: Lấy hành động thẩm định xong của ngân hàng SAU KHI nhận O_Returned
bank_decisions = df[df['concept:name'].isin(['A_Pending', 'A_Incomplete', 'A_Accepted', 'A_Denied', 'A_Cancelled'])]

# Nối với o_returned để đảm bảo hành động này xảy ra sau O_Returned
bank_decisions = bank_decisions.merge(o_returned, on='case:concept:name')
bank_decisions = bank_decisions[bank_decisions['time:timestamp'] >= bank_decisions['time_returned']]

# Lấy hành động đầu tiên sau O_Returned
time_bank_decision = bank_decisions.groupby('case:concept:name')['time:timestamp'].min().reset_index(name='time_bank_decision')

# Merge tất cả lại
wait_df = o_sent.merge(o_returned, on='case:concept:name', how='inner')
wait_df = wait_df.merge(time_bank_decision, on='case:concept:name', how='inner')

# Lọc bỏ những dữ liệu lỗi (âm thời gian)
wait_df = wait_df[(wait_df['time_sent'] <= wait_df['time_returned']) & (wait_df['time_returned'] <= wait_df['time_bank_decision'])]

print(f"Số lượng case hợp lệ tính toán: {len(wait_df)}")

In [ ]:
# 3. Tính toán Customer Wait Time và Bank Wait Time (đơn vị: Ngày)
wait_df['customer_wait_days'] = (wait_df['time_returned'] - wait_df['time_sent']).dt.total_seconds() / (24 * 3600)
wait_df['bank_wait_days'] = (wait_df['time_bank_decision'] - wait_df['time_returned']).dt.total_seconds() / (24 * 3600)

print("=== Thống kê Customer Wait Time (Ngày) ===")
print(wait_df['customer_wait_days'].describe())

print("\n=== Thống kê Bank Wait Time (Ngày) ===")
print(wait_df['bank_wait_days'].describe())

In [ ]:
# 4. Trực quan hóa so sánh trực quan
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Biểu đồ Boxplot so sánh độ phân tán
sns.boxplot(data=wait_df[['customer_wait_days', 'bank_wait_days']], ax=axes[0], palette='pastel')
axes[0].set_yscale('log') # Dùng log scale vì có các hồ sơ bị ngâm cả tháng (outliers)
axes[0].set_title('Phân bố thời gian chờ (Log Scale)', fontsize=14)
axes[0].set_ylabel('Số Ngày (log)')
axes[0].set_xticklabels(['Khách hàng giữ Offer', 'Ngân hàng chờ thẩm định'])

# Biểu đồ Barplot so sánh giá trị trung bình
mean_waits = [wait_df['customer_wait_days'].mean(), wait_df['bank_wait_days'].mean()]
sns.barplot(x=['Khách hàng giữ', 'Ngân hàng thẩm định'], y=mean_waits, ax=axes[1], palette='Set2')
axes[1].set_title('Trung bình thời gian chờ thực tế (Ngày)', fontsize=14)
axes[1].set_ylabel('Số Ngày')

# Hiển thị con số cụ thể trên cột
for i, v in enumerate(mean_waits):
    axes[1].text(i, v + 0.1, f"{v:.1f} ngày", ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# 5. Lấy LoanGoal từ dữ liệu gốc nối vào bảng Wait Time
app_info = df[df['concept:name'] == 'A_Create Application'][['case:concept:name', 'case:ApplicationType', 'case:LoanGoal']].drop_duplicates()

wait_df = wait_df.merge(app_info, on='case:concept:name', how='left')

# Trực quan hóa theo LoanGoal
top_goals = wait_df['case:LoanGoal'].value_counts().head(5).index # Chỉ lấy 5 mục đích vay phổ biến nhất
plot_df = wait_df[wait_df['case:LoanGoal'].isin(top_goals)].copy()

# Melting dataframe để vẽ barplot grouped dễ dàng hơn
melted_df = plot_df.melt(id_vars=['case:concept:name', 'case:LoanGoal'], 
                         value_vars=['customer_wait_days', 'bank_wait_days'],
                         var_name='Wait_Type', value_name='Days')

plt.figure(figsize=(14, 7))
sns.barplot(data=melted_df, x='case:LoanGoal', y='Days', hue='Wait_Type', palette='viridis', errorbar=None)
plt.title('So sánh Thời gian chờ Khách hàng vs Ngân hàng theo Mục đích vay', fontsize=15)
plt.ylabel('Trung bình (Ngày)')
plt.xlabel('Mục đích vay')
plt.xticks(rotation=15)
plt.legend(['Customer Wait (Ngày)', 'Bank Wait (Ngày)'])
plt.show()